In [41]:
import pandas as pd
import numpy as np
import random
from datetime import datetime, timedelta

# Set seed for reproducibility
np.random.seed(42)

# Configuration
num_rows = 10000
brands = ['Apple', 'Samsung', 'Xiaomi', 'Huawei', 'LG', 'Sony']
categories = ['electronics.smartphone', 'electronics.audio', 'appliances.kitchen']

# 1. Generate Timestamps
start_date = datetime(2026, 10, 1)
dates = [start_date + timedelta(hours=random.randint(0, 720)) for _ in range(num_rows)]

# 2. Generate Events based on a Funnel (View > Cart > Purchase)
# We simulate a 10% View-to-Cart and a 5% Cart-to-Purchase rate
event_types = np.random.choice(['view', 'cart', 'purchase'], num_rows, p=[0.85, 0.10, 0.05])

# 3. Create DataFrame
data = {
    'event_time': sorted(dates),
    'event_type': event_types,
    'product_id': [random.randint(100000, 999999) for _ in range(num_rows)],
    'category_code': [random.choice(categories) for _ in range(num_rows)],
    'brand': [random.choice(brands) for _ in range(num_rows)],
    'price': [round(random.uniform(50, 1500), 2) for _ in range(num_rows)],
    'user_id': [random.randint(500000000, 599999999) for _ in range(num_rows)],
    'user_session': [random.randint(1000, 5000) for _ in range(num_rows)]
}

df = pd.DataFrame(data)

# Save to CSV
df.to_csv('Simulated_Ecommerce_Data.csv', index=False)
print("Dataset 'Simulated_Ecommerce_Data.csv' created successfully!")
df.head()

Dataset 'Simulated_Ecommerce_Data.csv' created successfully!


,event_time,event_type,product_id,category_code,brand,price,user_id,user_session
0,2026-10-01,view,698613,electronics.smartphone,Huawei,981.30,567552683,3783
1,2026-10-01,purchase,695439,electronics.smartphone,Sony,1093.52,545095661,3402
2,2026-10-01,view,221299,electronics.smartphone,LG,1273.37,595957123,3027
3,2026-10-01,view,601904,appliances.kitchen,Huawei,323.22,543064124,1671
4,2026-10-01,view,148440,electronics.smartphone,Huawei,1136.84,509731089,3723


In [42]:
df['event_time'] = pd.to_datetime(df['event_time'])
df['day'] = df['event_time'].dt.day

daily_trends = df.groupby(['day', 'event_type']).size().unstack(fill_value=0)
# This gives you the X (Day) and Y (Count) values for your Area Chart

In [43]:
brand_share = df[df['event_type'] == 'view']['brand'].value_counts()
# Use this for your Top 3 "Traffic" sources in the donut chart

In [44]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

# 1. Load the simulated dataset
df = pd.read_csv('Simulated_Ecommerce_Data.csv')

# 2. Structure the Funnel Data
# We count unique users at each stage to identify drop-offs
funnel_counts = df.groupby('event_type')['user_id'].nunique().reindex(['view', 'cart', 'purchase']).reset_index()
funnel_counts.columns = ['Stage', 'Unique Users']

# 3. Calculate Conversion Rates (Task Requirement)
view_to_cart = (funnel_counts.iloc[1,1] / funnel_counts.iloc[0,1]) * 100
cart_to_purchase = (funnel_counts.iloc[2,1] / funnel_counts.iloc[1,1]) * 100
overall_conv = (funnel_counts.iloc[2,1] / funnel_counts.iloc[0,1]) * 100

# 4. Create the Funnel Visual (Modern UI Style)
fig_funnel = px.funnel(funnel_counts, x='Unique Users', y='Stage',
                       title='Customer Conversion Funnel',
                       color_discrete_sequence=['#8E54E9']) # Matching Lector Purple
fig_funnel.show()

# 5. Create the Wavy Trend Line (Spline Area Chart)
df['event_time'] = pd.to_datetime(df['event_time'])
df_trend = df[df['event_type']=='purchase'].resample('D', on='event_time').size().reset_index(name='Sales')

fig_trend = px.area(df_trend, x='event_time', y='Sales', title='Sales Trend Over Time',
                    line_shape='spline') # This creates the smooth 'wavy' look
fig_trend.update_traces(fillcolor='rgba(142, 84, 233, 0.2)', line_color='#8E54E9')
fig_trend.show()

In [45]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# 1. Load Data
df = pd.read_csv('Simulated_Ecommerce_Data.csv')
df['event_time'] = pd.to_datetime(df['event_time'])

# 2. Setup the "Lector" Theme Colors
colors = {
    'purple': '#8E54E9',
    'pink': '#FF4B2B',
    'orange': '#F6D365',
    'bg_light': '#F8F9FD',
    'grid_color': '#EAEAEA'
}

# --- CHART 1: The Wavy Spline Area Chart (Center Visual) ---
df_trend = df[df['event_type']=='purchase'].resample('D', on='event_time').size().reset_index(name='Sales')

fig_trend = go.Figure()
fig_trend.add_trace(go.Scatter(
    x=df_trend['event_time'], y=df_trend['Sales'],
    mode='lines',
    line=dict(width=4, color=colors['purple'], shape='spline'), # 'spline' makes it wavy
    fill='tozeroy',
    fillcolor='rgba(142, 84, 233, 0.1)' # Soft purple glow
))

fig_trend.update_layout(
    title="Overview of Latest Sales",
    plot_bgcolor='white',
    paper_bgcolor='white',
    font=dict(family="Arial", size=12, color="#7f7f7f"),
    xaxis=dict(showgrid=False),
    yaxis=dict(gridcolor=colors['grid_color']),
    margin=dict(l=40, r=40, t=60, b=40)
)

# --- CHART 2: The Traffic Ring (Donut Chart) ---
brand_data = df['brand'].value_counts().head(5)
fig_donut = px.pie(
    brand_data, values=brand_data.values, names=brand_data.index,
    hole=0.7, # Makes it a thin ring like the image
    color_discrete_sequence=[colors['purple'], colors['pink'], colors['orange'], '#4facfe', '#00f2fe']
)

fig_donut.update_layout(
    showlegend=True,
    annotations=[dict(text='Traffic', x=0.5, y=0.5, font_size=20, showarrow=False)],
    margin=dict(l=20, r=20, t=20, b=20)
)

# --- CHART 3: The Marketing Funnel (Task 3 Requirement) ---
funnel_data = df['event_type'].value_counts().reindex(['view', 'cart', 'purchase']).reset_index()
fig_funnel = go.Figure(go.Funnel(
    y = funnel_data['event_type'],
    x = funnel_data['count'],
    marker = {"color": [colors['purple'], colors['pink'], colors['orange']]}
))

fig_funnel.update_layout(title="Conversion Drop-offs", margin=dict(l=40, r=40, t=60, b=40))

# Show the visuals
fig_trend.show()
fig_donut.show()
fig_funnel.show()

In [46]:
# --- PERFORMANCE COMPARISON BY BRAND ---
# 1. Calculate Conversion Rate per Brand
brand_stats = df.groupby('brand').agg(
    Total_Views=('event_type', lambda x: (x=='view').sum()),
    Purchases=('event_type', lambda x: (x=='purchase').sum())
)
brand_stats['Conv_Rate %'] = (brand_stats['Purchases'] / brand_stats['Total_Views']) * 100
brand_stats = brand_stats.sort_values(by='Conv_Rate %', ascending=False)

# 2. Visualizing Performance Comparison
fig_brand = px.bar(brand_stats.reset_index(), x='brand', y='Conv_Rate %',
                   title='Conversion Performance by Brand',
                   color='Conv_Rate %', color_continuous_scale='Purp')
fig_brand.show()

# --- AUTOMATED INSIGHT GENERATION ---
# Logic to identify the biggest drop-off point
view_count = len(df[df['event_type'] == 'view'])
cart_count = len(df[df['event_type'] == 'cart'])
purchase_count = len(df[df['event_type'] == 'purchase'])

drop_off_view_to_cart = (1 - (cart_count / view_count)) * 100
drop_off_cart_to_purchase = (1 - (purchase_count / cart_count)) * 100

print("-" * 50)
print("🎯 TASK 3: FINAL ANALYTICS REPORT")
print("-" * 50)
print(f"1. DROP-OFF ANALYSIS:")
print(f"   - View-to-Cart Drop-off: {drop_off_view_to_cart:.2f}%")
print(f"   - Cart-to-Purchase Drop-off: {drop_off_cart_to_purchase:.2f}%")

print(f"\n2. TOP PERFORMING CHANNEL (BRAND):")
print(f"   - {brand_stats.index[0]} leads with a {brand_stats.iloc[0,2]:.2f}% conversion rate.")

print("\n3. ACTIONABLE RECOMMENDATIONS:")
if drop_off_cart_to_purchase > 50:
    print("   - [HIGH PRIORITY] Implement Abandoned Cart Recovery: 50%+ users leave at checkout.")
if drop_off_view_to_cart > 80:
    print("   - [UI/UX] Improve Product Pages: High traffic but low 'Add to Cart' intent.")
print(f"   - [MARKETING] Reallocate budget to {brand_stats.index[0]}: Highest ROI potential.")
print("-" * 50)

--------------------------------------------------
🎯 TASK 3: FINAL ANALYTICS REPORT
--------------------------------------------------
1. DROP-OFF ANALYSIS:
   - View-to-Cart Drop-off: 88.61%
   - Cart-to-Purchase Drop-off: 51.33%

2. TOP PERFORMING CHANNEL (BRAND):
   - Xiaomi leads with a 6.19% conversion rate.

3. ACTIONABLE RECOMMENDATIONS:
   - [HIGH PRIORITY] Implement Abandoned Cart Recovery: 50%+ users leave at checkout.
   - [UI/UX] Improve Product Pages: High traffic but low 'Add to Cart' intent.
   - [MARKETING] Reallocate budget to Xiaomi: Highest ROI potential.
--------------------------------------------------
